# War and conflict

In [ ]:
import json
import polars as pl

## Get data

In [ ]:
df = pl.read_parquet('../data/Table1_Fact/FIN/*.parquet')

In [ ]:
df.schema

In [ ]:
# Number of rows
df.height

In [ ]:
# Annotate session year
df = df.with_columns(pl.col('session_date').dt.year().alias('session_year'))

# Number of rows per year
df.group_by('session_year').agg(pl.len().alias('nb_rows_per_year')).sort('session_year')

In [ ]:
# Number of rows per debate topic
df.group_by('debate_topic').agg(pl.len().alias('nb_rows_per_topic')).sort(
    'debate_topic'
)

## Merge data

In [ ]:
df = (
    df.join(
        pl.read_parquet('../data/Table2_People/Master_People.parquet'),
        on=['speaker_id', 'country'],
        how='left',
    )
    .join(
        pl.read_parquet('../data/Table4_Affiliations/Master_Affiliations.parquet').with_columns(
            pl.when(pl.col('start_date').is_not_null())
            .then(pl.col('start_date').str.to_date('%Y-%m-%d', strict=False))
            .otherwise(None)
            .alias('start_date'),
            pl.when(pl.col('end_date').is_not_null())
            .then(pl.col('end_date').str.to_date('%Y-%m-%d', strict=False))
            .otherwise(None)
            .alias('end_date'),
        ),
        on=['speaker_id', 'country'],
        how='left',
    )
    .filter(
        (pl.col('session_date') >= pl.col('start_date'))
        & (
            pl.col('end_date').is_null()
            | (pl.col('session_date') <= pl.col('end_date'))
        )
    )
    .unique()
    .join(
        pl.read_parquet('../data/Table3_Orgs/Master_Organizations.parquet'),
        on=['org_id', 'country'],
        how='left',
    )
)


df

## Filter data

In [ ]:
def get_results(filtered_df):

    results = (
        filtered_df.with_columns(
            pl.struct(
                [
                    pl.col('session_date'),
                    pl.col('debate_topic'),
                    pl.col('country'),
                    pl.col('speaker_id'),
                    pl.col('party_left_right_orientation'),
                    pl.col('sentence_content_previous'),
                    pl.col('sentence_content_current'),
                    pl.col('sentence_content_next'),
                ]
            ).alias('sentence_with_context')
        )
        .unique()
        .sort('session_date')
        .group_by([pl.col('entity_category'), pl.col('entity_content')])
        .agg(pl.col('sentence_with_context').alias('sentences_with_context'))
    )

    # Deduplicate 'party_left_right_orientation' in 'sentences_with_context'
    return (
        results.explode('sentences_with_context')
        .unnest('sentences_with_context')
        .group_by(
            [
                'entity_category',
                'entity_content',
                'session_date',
                'debate_topic',
                'country',
                'speaker_id',
                'sentence_content_previous',
                'sentence_content_current',
                'sentence_content_next'
            ]
        )
        .agg(pl.col('party_left_right_orientation').unique(maintain_order=True))
        .with_columns(
            pl.struct(
                'session_date',
                'debate_topic',
                'country',
                'speaker_id',
                'party_left_right_orientation',
                'sentence_content_previous',
                'sentence_content_current',
                'sentence_content_next'
            ).alias('sentences_with_context')
        )
        .group_by(['entity_category', 'entity_content'])
        .agg(pl.col('sentences_with_context'))
        .with_columns(pl.col('sentences_with_context').list.len().alias('nb_sentences'))
        .sort('nb_sentences', descending=True)
    )

In [ ]:
# Filter dataset to remove some historical conflicts
HISTORICAL_CONFLICTS = ['world war', 'cold war', 'winter war', 'homeland war']
df = df.filter(
    ~pl.col('entity_content').str.contains_any(
        HISTORICAL_CONFLICTS, ascii_case_insensitive=True
    ),
)

### Keywords

In [ ]:
# Look for a particular keyword in "sentence_content_current"
def search_keyword(keyword):
    pattern = rf'(?i){keyword}'
    return get_results(
        df.filter(
            pl.col('sentence_content_current').str.contains(pattern),
        )
    )

In [ ]:
# Mentions of "war"
search_keyword(r'\bwar\b')

In [ ]:
# Mentions of "conflict"
search_keyword('conflict')

### Entities

In [ ]:
# Look for a particular entity in "entity_content"
def search_entity(entity):
    pattern = rf'(?i){entity}'
    return get_results(df.filter(pl.col('entity_content').str.contains(pattern)))

In [ ]:
# Mentions of "ukrain"
search_entity('ukrain')

In [ ]:
# Mentions of "afghan"
search_entity('afghan')

### Keywords and entities

In [ ]:
# Look for a particular keyword associated to a particular entity
def search_keyword_for_entity(keyword, entity):
    pattern_entity = rf'(?i){entity}'
    pattern_keyword = rf'(?i){keyword}'

    return get_results(
        df.filter(
            pl.col('entity_content').str.contains(pattern_entity),
            pl.col('sentence_content_current').str.contains(pattern_keyword),
        )
    )

In [ ]:
# Look for mentions of "war" keyword and "afghan" entity
search_keyword_for_entity(r'\bwar\b', 'afghan')

In [ ]:
# Look for mentions of "war" keyword and "ukrain" entity
search_keyword_for_entity(r'\bwar\b', 'ukrain')

### Specific conflicts

In [ ]:
OUTPUT_PATH = './output.json'

COUNTRIES_IN_CONFLICT = [
    'Afghanistan',
    'Sudan',
    'Thailand',
    'Iran',
    'Mexico',
    'Somalia',
    'Russia',
    'Tajikistan',
    'Syria',
    'Egypt',
    'Central African Republic',
    'Congo',
    'Iraq',
    'Libya',
    'Yemen',
    'Ukraine',
    'Burkina Faso',
    'Niger',
    'Armenia',
    'Philippines',
    'Myanmar',
    'Cameroon',
    'Mozambique',
    'Mali',
    'Uganda',
    'Ethiopia',
    'Israel',
    'Palestine',
]

results = {}

for country in COUNTRIES_IN_CONFLICT:
    results[country] = search_keyword_for_entity(r'\bwar\b|conflict', country).write_json()

with open(OUTPUT_PATH, 'w', encoding='utf-8') as file:
    json.dump(results, file, ensure_ascii=False, indent=4)